# Tutorial de IA responsable con Fairlearn y Aequitas

Este notebook muestra un flujo completo para **entrenar, evaluar y auditar un modelo de clasificación** sobre el conjunto de datos COMPAS.

Se utilizan dos bibliotecas complementarias:

- **Fairlearn**: aplica técnicas de preprocesamiento para reducir la dependencia entre los datos y los atributos sensibles.
- **Aequitas**: calcula métricas por grupo, disparidades respecto a grupos de referencia y evaluaciones de equidad.

<!-- Al terminar el tutorial se habrán realizado estas etapas:

1. Carga y filtrado de los datos.
2. Preparación de características y contexto de auditoría.
3. Mitigación opcional mediante Fairlearn.
4. Entrenamiento con validación cruzada estratificada.
5. Evaluación del rendimiento predictivo.
6. Auditoría de disparidades y equidad con Aequitas. -->

## Estructura del flujo

El proyecto está dividido en tres módulos:

- `preprocessing.py`: carga los datos, aplica los filtros, codifica las variables y ejecuta el método de Fairlearn elegido.
- `train_model.py`: entrena el modelo mediante validación cruzada y genera predicciones fuera de muestra.
- `auditar.py`: entrega esas predicciones a Aequitas y calcula los resultados de equidad.

Separar el proceso en módulos permite reutilizar las funciones, probar distintos modelos y comparar métodos de mitigación sin duplicar código.

In [1]:
from pathlib import Path
import warnings

import pandas as pd
from IPython.display import display

from preprocessing import Preprocesar
from train_model import train_model
from auditar import auditar_modelo

warnings.filterwarnings("ignore")

## 3. Conjunto de datos

El archivo utilizado es `compas-scores-two-years.csv`, que contiene información demográfica, antecedentes, datos del cargo y la variable de reincidencia a dos años.

In [2]:
csv_path = Path("data") / "raw" / "compas-scores-two-years.csv"

print(f"Conjunto de datos: {csv_path.resolve()}")

Conjunto de datos: D:\Escuela\Maestría\Proyecto Tecnologico\archivos_IA_responsable_comentados\data\raw\compas-scores-two-years.csv


## 4. Preprocesamiento y mitigación con Fairlearn

La función `Preprocesar` realiza las siguientes operaciones:

1. Lee el archivo CSV.
2. Aplica filtros para conservar observaciones válidas del análisis de COMPAS.
3. Selecciona las variables usadas por el modelo.
4. Conserva por separado `race`, `sex` y `age_cat` con sus valores originales para la auditoría.
5. Imputa valores numéricos ausentes con la mediana.
6. Codifica las variables categóricas con `OrdinalEncoder`.
7. Aplica opcionalmente un método de Fairlearn.

### Métodos disponibles

- `None`: no aplica mitigación; sirve como modelo base.
- `"CorrelationRemover"`: transforma las variables no sensibles para retirar la correlación lineal con `race` y `sex`.
- `"PrototypeRepresentationLearner"`: aprende una nueva representación basada en prototipos para conservar utilidad predictiva y reducir diferencias entre grupos.

El preprocesamiento produce dos tablas:

- `compas_preprocessed`: variables numéricas listas para entrenar y la etiqueta `two_year_recid`.
- `compas_audit_context`: identificador, atributos protegidos y etiqueta real sin codificar.

In [3]:
# Cambia este valor para comparar diferentes estrategias de mitigación.
PREPROCESSING_METHOD = "PrototypeRepresentationLearner"
# Alternativas:
# PREPROCESSING_METHOD = "CorrelationRemover"
# PREPROCESSING_METHOD = None

compas_preprocessed, compas_audit_context = Preprocesar(
    csv_path,
    preprocessing_method=PREPROCESSING_METHOD,
)

### Inspección de las salidas

Es importante comprobar que ambas tablas tengan el mismo número de filas. La correspondencia fila a fila permite unir correctamente cada predicción con sus atributos protegidos durante la auditoría.

Las columnas del conjunto procesado pueden cambiar según el método:

- Sin mitigación se conservan las características codificadas.
- `CorrelationRemover` elimina las columnas sensibles del resultado transformado.
- `PrototypeRepresentationLearner` genera columnas nuevas llamadas `prototype_0`, `prototype_1`, etc.

In [4]:
print("Método de preprocesamiento:", PREPROCESSING_METHOD)
print("Forma del conjunto para entrenamiento:", compas_preprocessed.shape)
print("Forma del contexto de auditoría:", compas_audit_context.shape)

print("\nDatos preparados para entrenamiento:")
display(compas_preprocessed.head())

print("\nContexto conservado para la auditoría:")
display(compas_audit_context.head())

Método de preprocesamiento: PrototypeRepresentationLearner
Forma del conjunto para entrenamiento: (6172, 3)
Forma del contexto de auditoría: (6172, 5)

Datos preparados para entrenamiento:


,prototype_0,prototype_1,two_year_recid
0,0.222960,0.777040,0
1,0.674250,0.325750,1
2,0.832128,0.167872,1
3,0.492094,0.507906,0
4,0.473768,0.526232,1



Contexto conservado para la auditoría:


,entity_id,race,sex,age_cat,label_value
0,1,Other,Male,Greater than 45,0
1,3,African-American,Male,25 - 45,1
2,4,African-American,Male,Less than 25,1
3,7,Other,Male,25 - 45,0
4,8,Caucasian,Male,25 - 45,1


## 5. Entrenamiento con validación cruzada

La función `train_model` usa por defecto una **regresión logística** y una validación cruzada `StratifiedKFold` de 10 particiones.

La validación estratificada conserva aproximadamente la proporción de las clases en cada fold. En cada iteración:

1. Se entrena un modelo con nueve particiones.
2. Se predice la partición restante.
3. Se calculan `accuracy`, `precision`, `recall` y `F1`.
4. Se guarda el contexto de las observaciones de prueba junto con sus predicciones.

Al finalizar, cada persona tiene una predicción obtenida por un modelo que **no fue entrenado con esa misma observación**. Esto evita auditar predicciones hechas directamente sobre los datos de entrenamiento.

In [ ]:
result, audit_predictions = train_model(
    compas_preprocessed,
    compas_audit_context,
    model_name=f"LR_{PREPROCESSING_METHOD or 'sin_mitigacion'}",
    feature_set=PREPROCESSING_METHOD or "sin_mitigacion",
    n_splits=10,
    random_state=42,
)

### Métricas predictivas

El resultado incluye:

- `metrics`: promedio de cada métrica entre los folds.
- `metrics_std`: desviación estándar entre folds.
- `fold_metrics`: resultados individuales de cada partición.

Estas métricas miden la utilidad predictiva general, pero **no indican por sí solas si el modelo se comporta de forma similar entre grupos**. Esa parte se analiza después con Aequitas.

In [ ]:
metrics_summary = pd.DataFrame(
    [
        result["metrics"],
        result["metrics_std"],
    ],
    index=[
        "Media",
        "Desviación estándar",
    ],
)

fold_metrics = pd.DataFrame(result["fold_metrics"]).set_index("fold")

print("Resumen de rendimiento:")
display(metrics_summary)

print("\nMétricas por fold:")
display(fold_metrics)

## 6. Tabla de predicciones para auditoría

`audit_predictions` contiene una fila por observación y utiliza el formato requerido por Aequitas.

Columnas principales:

- `entity_id`: identificador de la observación.
- `model_id`: nombre del modelo auditado.
- `feature_set`: estrategia de preprocesamiento utilizada.
- `fold`: partición que produjo la predicción.
- `race`, `sex`, `age_cat`: atributos protegidos originales.
- `label_value`: valor real de reincidencia.
- `score`: clase predicha por el modelo, 0 o 1.

En este proyecto, `score` representa una **decisión binaria**, no una probabilidad.

In [ ]:
print("Número de predicciones para auditoría:", len(audit_predictions))
display(audit_predictions.head(10))

## 7. Auditoría con Aequitas

La función `auditar_modelo` ejecuta tres niveles de análisis:

### 7.1 Métricas absolutas por grupo

`Group.get_crosstabs` calcula métricas como:

- `prev`: prevalencia real de la clase positiva.
- `pprev`: proporción de predicciones positivas.
- `tpr`: tasa de verdaderos positivos.
- `tnr`: tasa de verdaderos negativos.
- `fpr`: tasa de falsos positivos.
- `fnr`: tasa de falsos negativos.
- `precision`: proporción de predicciones positivas correctas.

### 7.2 Disparidades

`Bias.get_disparity_predefined_groups` divide la métrica de cada grupo entre la métrica del grupo de referencia.

Los grupos de referencia predeterminados son:

- `race`: `Caucasian`
- `sex`: `Male`
- `age_cat`: `25 - 45`

Una disparidad cercana a `1` indica comportamiento semejante al grupo de referencia. Una disparidad no demuestra por sí sola discriminación causal; señala diferencias que deben investigarse.

### 7.3 Evaluación de equidad

Con `tau=0.80`, Aequitas aplica una regla de paridad equivalente al intervalo aproximado `[0.80, 1.25]`. Una razón fuera de ese intervalo se marca como una posible falla de paridad.

El parámetro `alpha=0.05` se utiliza para comprobar significancia estadística.

In [ ]:
resultado = auditar_modelo(
    audit_predictions,
    tau=0.80,
    alpha=0.05,
)

## 8. Selección de columnas relevantes

Aequitas produce tablas con muchas columnas. Para facilitar la lectura se crean vistas reducidas sin modificar los resultados completos guardados en `resultado`.

- La primera vista muestra métricas absolutas.
- La segunda muestra disparidades respecto al grupo de referencia.
- La tercera resume diferentes criterios de paridad.

In [ ]:
ABSOLUTE_COLUMNS = [
    "model_id",
    "attribute_name",
    "attribute_value",
    "group_size",
    "prev",
    "pprev",
    "tpr",
    "tnr",
    "fpr",
    "fnr",
    "fdr",
    "for",
    "precision",
    "npv",
]

DISPARITY_COLUMNS = [
    "model_id",
    "attribute_name",
    "attribute_value",
    "group_size",
    "fpr_disparity",
    "fnr_disparity",
    "pprev_disparity",
    "fdr_disparity",
    "precision_disparity",
    "fpr_ref_group_value",
    "fnr_ref_group_value",
]

FAIRNESS_COLUMNS = [
    "model_id",
    "attribute_name",
    "attribute_value",
    "group_size",
    "FPR Parity",
    "FNR Parity",
    "Statistical Parity",
    "Impact Parity",
    "Equalized Odds",
    "TypeI Parity",
    "TypeII Parity",
    "Unsupervised Fairness",
    "Supervised Fairness",
]

group_metrics_view = resultado["group_metrics"].loc[:, ABSOLUTE_COLUMNS]
disparities_view = resultado["disparities"].loc[:, DISPARITY_COLUMNS]
group_fairness_view = resultado["group_fairness"].loc[:, FAIRNESS_COLUMNS]

## 9. Resultados por grupo

Al interpretar estas tablas conviene revisar simultáneamente:

1. El tamaño del grupo (`group_size`), porque grupos pequeños producen estimaciones más inestables.
2. La métrica absoluta, para conocer la magnitud real del error.
3. La disparidad, para compararla con el grupo de referencia.
4. La significancia estadística, disponible en la tabla completa de disparidades.
5. El contexto del problema, ya que no existe una única métrica de equidad adecuada para todos los usos.

Por ejemplo, una diferencia en `fpr` indica que un grupo recibe falsos positivos con mayor frecuencia. Una diferencia en `fnr` indica que el modelo omite positivos reales con distinta frecuencia.

In [ ]:
print("Métricas absolutas por grupo:")
display(group_metrics_view)

print("\nDisparidades respecto al grupo de referencia:")
display(disparities_view)

print("\nEvaluaciones de equidad por grupo:")
display(group_fairness_view)

## 10. Resúmenes por atributo y del modelo completo

`attribute_fairness` agrega los resultados de todos los valores de un atributo, por ejemplo todos los grupos de `race`.

`overall_fairness` resume la evaluación del modelo completo. Este resumen es útil para identificar rápidamente si existe alguna falla, pero no sustituye la inspección de las métricas y disparidades específicas.

In [ ]:
print("Resumen de equidad por atributo:")
display(resultado["attribute_fairness"])

print("\nResumen global de equidad:")
display(resultado["overall_fairness"])

## 11. Cómo comparar estrategias

Para evaluar el efecto de Fairlearn, se recomienda ejecutar el notebook con:

1. `PREPROCESSING_METHOD = None`
2. `PREPROCESSING_METHOD = "CorrelationRemover"`
3. `PREPROCESSING_METHOD = "PrototypeRepresentationLearner"`

Después se deben comparar dos dimensiones:

- **Utilidad predictiva**: cambios en `accuracy`, `precision`, `recall` y `F1`.
- **Equidad**: cambios en `fpr_disparity`, `fnr_disparity`, `pprev_disparity` y los indicadores de paridad.

Una técnica de mitigación no garantiza que todas las métricas mejoren simultáneamente. Es normal encontrar compromisos entre rendimiento, distintos criterios de equidad y grupos diferentes.

## 12. Limitaciones del análisis

- Las métricas describen asociaciones observadas; no prueban causalidad.
- Los resultados dependen de la calidad de las etiquetas y del proceso que generó los datos.
- La codificación ordinal asigna números a categorías sin implicar que exista una distancia natural entre ellas.
- Los grupos pequeños pueden producir métricas variables o poco confiables.
- La selección del grupo de referencia y del umbral `tau` afecta la evaluación.
- La equidad técnica debe complementarse con análisis jurídico, social y del contexto de uso.